In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ==================== 修复：导入缺失的 CosineAnnealingLR ====================
from torch.optim.lr_scheduler import CosineAnnealingLR

# ==================== 1. 数据增强策略 ====================
class AdvancedSimCLRTransform:
    def __init__(self, image_size=32, strong_aug=True):
        self.image_size = image_size
        self.strong_aug = strong_aug
        # CIFAR-10 标准化参数
        self.mean = [0.4914, 0.4822, 0.4465]
        self.std = [0.2470, 0.2435, 0.2616]
        
    def __call__(self, x):
        if self.strong_aug:
            transform1 = transforms.Compose([
                transforms.RandomResizedCrop(size=self.image_size, scale=(0.08, 1.0)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomApply([transforms.ColorJitter(0.8,0.8,0.8,0.2)], p=0.8),
                transforms.RandomGrayscale(p=0.2),
                transforms.ToTensor(),
                transforms.Normalize(self.mean, self.std),
            ])
            transform2 = transforms.Compose([
                transforms.RandomResizedCrop(size=self.image_size, scale=(0.08, 1.0)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
                transforms.RandomGrayscale(p=0.2),
                transforms.ToTensor(),
                transforms.Normalize(self.mean, self.std),
            ])
        else:
            transform1 = transforms.Compose([
                transforms.RandomResizedCrop(self.image_size, scale=(0.2,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(self.mean, self.std),
            ])
            transform2 = transform1
        return transform1(x), transform2(x)

# ==================== 2. 模型架构 ====================
class ImprovedResNetBackbone(nn.Module):
    def __init__(self, base_encoder='resnet50', pretrained=False):
        super().__init__()
        if base_encoder == 'resnet18':
            encoder = torchvision.models.resnet18
            self.global_dim = 512
            self.local_dim = 256
        elif base_encoder == 'resnet50':
            encoder = torchvision.models.resnet50
            self.global_dim = 2048
            self.local_dim = 1024
        else:
            raise ValueError(f"Unsupported base_encoder: {base_encoder}")
        
        try:
            self.backbone = encoder(weights="DEFAULT") if pretrained else encoder(weights=None)
        except:
            self.backbone = encoder(pretrained=pretrained)
        
        # CIFAR-10 适配：移除大核池化，适配 32x32
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Identity()
        
        self.layer1 = self.backbone.layer1
        self.layer2 = self.backbone.layer2
        self.layer3 = self.backbone.layer3
        self.layer4 = self.backbone.layer4
    
    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x3 = self.layer3(x)
        x4 = self.layer4(x3)
        
        x_global = nn.functional.adaptive_avg_pool2d(x4, (1,1)).flatten(1)
        x_local = nn.functional.adaptive_avg_pool2d(x3, (4,4))
        return x_global, x_local

class ImprovedProjectionHead(nn.Module):
    def __init__(self, input_dim, output_dim=256, hidden_dim=2048):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x):
        return self.mlp(x)

class ImprovedSimCLRModel(nn.Module):
    def __init__(self, backbone, global_projector, local_projector):
        super().__init__()
        self.encoder = backbone
        self.global_projector = global_projector
        self.local_projector = local_projector
    
    def forward(self, x):
        global_feat, local_feat = self.encoder(x)
        
        z_global = self.global_projector(global_feat)
        z_global = nn.functional.normalize(z_global, dim=1)
        
        local_feat = local_feat.flatten(1)
        z_local = self.local_projector(local_feat)
        z_local = nn.functional.normalize(z_local, dim=1)
        
        return z_global, z_local, global_feat

# ==================== 3. 损失函数：联合损失 + 系数 ====================
def improved_simclr_loss(z1, z2, temperature=0.07):
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.T) / temperature
    
    labels = torch.cat([torch.arange(B) for _ in range(2)], dim=0).to(z.device)
    labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
    mask = torch.eye(2*B, device=z.device).bool()
    labels = labels.masked_fill(mask, 0)
    
    logits = sim - torch.max(sim, dim=1, keepdim=True)[0].detach()
    exp_logits = torch.exp(logits)
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))
    loss = -(labels * log_prob).sum(1) / labels.sum(1)
    return loss.mean()

def combined_feat_loss(z1_g, z2_g, z1_l, z2_l, temp=0.07, alpha=0.7, beta=0.3):
    loss_g = improved_simclr_loss(z1_g, z2_g, temp)
    loss_l = improved_simclr_loss(z1_l, z2_l, temp)
    total = alpha * loss_g + beta * loss_l
    return total, loss_g, loss_l

# ==================== 4. 训练 ====================
def train_epoch(model, loader, opt, device, epoch, alpha=0.7, beta=0.3, use_amp=True):
    model.train()
    total_loss = 0
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for v1, v2, _ in pbar:
        v1, v2 = v1.to(device), v2.to(device)
        opt.zero_grad()
        
        if use_amp:
            with torch.cuda.amp.autocast():
                z1_g, z1_l, _ = model(v1)
                z2_g, z2_l, _ = model(v2)
                loss, loss_g, loss_l = combined_feat_loss(z1_g,z2_g,z1_l,z2_l,0.07, alpha, beta)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            z1_g, z1_l, _ = model(v1)
            z2_g, z2_l, _ = model(v2)
            loss, loss_g, loss_l = combined_feat_loss(z1_g,z2_g,z1_l,z2_l,0.07, alpha, beta)
            loss.backward()
            opt.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss":f"{loss:.3f}", "g":f"{loss_g:.2f}", "l":f"{loss_l:.2f}"})
    return total_loss / len(loader)

# ==================== 5. 线性评估 ====================
def linear_evaluation_simple(model, train_loader, test_loader, num_classes=10, epochs=20, device='cuda'):
    model.eval()
    for p in model.parameters(): p.requires_grad = False
    
    linear = nn.Sequential(
        nn.Linear(2048, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    ).to(device)
    
    opt = optim.AdamW(linear.parameters(), lr=0.01, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_acc = 0
    
    for ep in range(epochs):
        linear.train()
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            with torch.no_grad():
                _, _, feat = model(x)
            loss = criterion(linear(feat), y)
            opt.zero_grad(); loss.backward(); opt.step()
        
        linear.eval()
        cor, tot = 0,0
        with torch.no_grad():
            for x,y in test_loader:
                x,y = x.to(device), y.to(device)
                _, _, feat = model(x)
                cor += (linear(feat).argmax(1)==y).sum()
                tot += y.size(0)
        acc = 100.*cor/tot
        if acc>best_acc: best_acc=acc
        print(f"Linear Ep{ep+1} | Acc: {acc:.2f}% | Best: {best_acc:.2f}%")
        scheduler.step()
    return best_acc

# ==================== 主函数 ====================
def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    config = {
        'batch_size': 128,
        'lr': 3e-4,
        'epochs': 50,
        'alpha': 0.7,
        'beta': 0.3,
        'base_encoder':'resnet50',
        'pretrained':True,
        'use_amp':True,
    }
    
    os.makedirs('checkpoints_comb', exist_ok=True)
    
    # ====================== CIFAR-10 数据集 ======================
    image_size = 32
    class DualViewDataset:
        def __init__(self, ds, tfm): self.ds = ds; self.tfm = tfm
        def __len__(self): return len(self.ds)
        def __getitem__(self,i): img,l=self.ds[i]; v1,v2=self.tfm(img); return v1,v2,l
    
    # 自监督训练用训练集（无标签）
    train_ds = DualViewDataset(
        datasets.CIFAR10('./data', train=True, download=True),
        AdvancedSimCLRTransform(image_size, True)
    )
    train_loader = DataLoader(train_ds, config['batch_size'], shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    
    # 模型
    backbone = ImprovedResNetBackbone(config['base_encoder'], config['pretrained'])
    global_proj = ImprovedProjectionHead(backbone.global_dim, 256, 2048)
    local_proj = ImprovedProjectionHead(backbone.local_dim*4*4, 256, 2048)
    model = ImprovedSimCLRModel(backbone, global_proj, local_proj).to(device)
    
    # 优化
    opt = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(opt, T_max=config['epochs'])
    
    # 训练
    losses = []
    for ep in range(config['epochs']):
        loss = train_epoch(model, train_loader, opt, device, ep+1,
                          config['alpha'], config['beta'], config['use_amp'])
        scheduler.step()
        losses.append(loss)
        print(f"Epoch {ep+1}/{config['epochs']} | Loss: {loss:.4f} | LR: {opt.param_groups[0]['lr']:.6f}")
    
    torch.save(model.state_dict(), 'checkpoints_comb/final_comb_model.pth')
    
    # 线性评估
    eval_tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.4914, 0.4822, 0.4465],[0.2470, 0.2435, 0.2616])
    ])
    linear_train = DataLoader(datasets.CIFAR10('./data',train=True,transform=eval_tfm), 128, shuffle=True, num_workers=2)
    linear_test = DataLoader(datasets.CIFAR10('./data',train=False,transform=eval_tfm), 128, shuffle=False, num_workers=2)
    
    best_acc = linear_evaluation_simple(model, linear_train, linear_test, 10, 50, device)
    
    print("\n" + "="*50)
    print(f"FINAL BEST ACCURACY: {best_acc:.2f}%")
    print("="*50)

if __name__ == "__main__":
    torch.manual_seed(42)
    np.random.seed(42)
    main()

Using device: cuda
Files already downloaded and verified


Epoch 1: 100%|██████████| 390/390 [00:53<00:00,  7.30it/s, loss=3.209, g=3.18, l=3.29]


Epoch 1/50 | Loss: 3.7433 | LR: 0.000300


Epoch 2: 100%|██████████| 390/390 [00:51<00:00,  7.56it/s, loss=3.017, g=3.01, l=3.02]


Epoch 2/50 | Loss: 3.0398 | LR: 0.000299


...

Epoch 38/50 | Loss: 2.0539 | LR: 0.000041


Epoch 39: 100%|██████████| 390/390 [00:58<00:00,  6.63it/s, loss=2.146, g=2.15, l=2.14]


Epoch 39/50 | Loss: 2.0510 | LR: 0.000034


Epoch 40:  74%|███████▍  | 289/390 [00:42<00:14,  6.87it/s, loss=1.841, g=1.83, l=1.86]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Linear Ep45 | Acc: 89.16% | Best: 89.16%
Linear Ep46 | Acc: 89.01% | Best: 89.16%
Linear Ep47 | Acc: 89.06% | Best: 89.16%
Linear Ep48 | Acc: 89.18% | Best: 89.18%
Linear Ep49 | Acc: 89.29% | Best: 89.29%
Linear Ep50 | Acc: 89.17% | Best: 89.29%

FINAL BEST ACCURACY: 89.29%
